# Crop Disease Detection Using Convolutional Neural Networks (CNN)
### **CropDetect AI — Academic & Research Deep Learning Pipeline**
*Trained on the PlantVillage Benchmark Dataset (38 Classes, 14 Crops)*

This notebook provides the complete, end-to-end machine learning pipeline:
1. Environment Setup & Hardware Acceleration (NVIDIA GPU Verification)
2. Dataset Acquisition (PlantVillage from Kaggle / Direct Archive)
3. Dataset Exploration & Directory Traversal
4. Visualizing Sample Crop Leaf Images
5. Image Preprocessing & Standardization (224x224 RGB)
6. Stratified Train / Validation / Test Splitting (70% / 15% / 15%)
7. Data Augmentation Pipeline (Rotation, Zoom, Flips, Contrast)
8. Deep Convolutional Neural Network (CNN) Architecture
9. Model Compilation with Adam Optimizer & Categorical Crossentropy
10. Production Callbacks (EarlyStopping, ModelCheckpoint, ReduceLROnPlateau)
11. Model Training Execution
12. Training History Curves (Accuracy & Loss vs Epochs)
13. Confusion Matrix Heatmap Evaluation
14. Comprehensive Classification Report (Precision, Recall, F1-Score)
15. Single-Image Inference on Test Samples
16. Model Serialization (.keras and .h5)
17. Exporting Assets for the AgriLeaf / CropDetect AI Backend Service


## 1. Install & Import Dependencies

In [ ]:
# Verify GPU environment in Google Colab
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

# Set random seeds for reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)


## 2. Load PlantVillage Dataset

In [ ]:
# Option A: In Google Colab, download directly via Kaggle API or gdown / git
# To use Kaggle:
# !pip install -q kaggle
# from google.colab import files
# files.upload() # upload your kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d emmarex/plantdisease
# !unzip -q plantdisease.zip -d dataset/

DATASET_DIR = "dataset/PlantVillage"
if not os.path.exists(DATASET_DIR):
    print("Dataset directory not found. Please specify local path or run Kaggle download.")
    # For demonstration/offline fallback, create mock directory structure if needed:
    os.makedirs(DATASET_DIR, exist_ok=True)
else:
    print(f"Dataset found at: {DATASET_DIR}")


## 3. Explore Dataset Classes and Sample Distribution

In [ ]:
class_names = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f"Total Detected Classes: {len(class_names)}")
for i, cls in enumerate(class_names[:10]):
    count = len(os.listdir(os.path.join(DATASET_DIR, cls)))
    print(f"[{i+1:02d}] {cls}: {count} images")
if len(class_names) > 10:
    print(f"... and {len(class_names) - 10} more classes.")


## 4. Visualize Sample Leaf Images

In [ ]:
def plot_sample_images(dataset_dir, classes, n_samples=8):
    plt.figure(figsize=(16, 8))
    for i, cls in enumerate(classes[:n_samples]):
        cls_dir = os.path.join(dataset_dir, cls)
        img_files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('jpg', 'jpeg', 'png'))]
        if img_files:
            img_path = os.path.join(cls_dir, img_files[0])
            img = Image.open(img_path)
            plt.subplot(2, 4, i + 1)
            plt.imshow(img)
            clean_title = cls.replace("___", "\n").replace("_", " ")
            plt.title(clean_title, fontsize=9)
            plt.axis("off")
    plt.tight_layout()
    plt.show()

if os.path.exists(DATASET_DIR) and len(class_names) > 0:
    plot_sample_images(DATASET_DIR, class_names, 8)


## 5. Image Preprocessing & Hyperparameters

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CHANNELS = 3
NUM_CLASSES = len(class_names) if len(class_names) > 0 else 38
EPOCHS = 25
LEARNING_RATE = 0.0005

print(f"Image Target Resolution: {IMAGE_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Target Number of Classes: {NUM_CLASSES}")


## 6. Train / Validation / Test Split (70% / 15% / 15%)

In [ ]:
# Using tf.keras.utils.image_dataset_from_directory for efficient pipeline
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

# Split val_test_ds in half (15% validation, 15% test)
val_batches = tf.data.experimental.cardinality(val_test_ds) // 2
val_ds = val_test_ds.take(val_batches)
test_ds = val_test_ds.skip(val_batches)

# Prefetching for GPU optimization
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 7. Data Augmentation Pipeline

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.08, 0.08),
], name="data_augmentation")


## 8. Deep Convolutional Neural Network (CNN) Architecture

In [ ]:
def build_crop_cnn(input_shape=(224, 224, 3), num_classes=38):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        data_augmentation,
        tf.keras.layers.Rescaling(1.0 / 255.0),

        # Conv Block 1
        tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu', name="conv1"),
        tf.keras.layers.BatchNormalization(name="bn1"),
        tf.keras.layers.MaxPooling2D((2, 2), name="pool1"),

        # Conv Block 2
        tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu', name="conv2"),
        tf.keras.layers.BatchNormalization(name="bn2"),
        tf.keras.layers.MaxPooling2D((2, 2), name="pool2"),

        # Conv Block 3
        tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu', name="conv3"),
        tf.keras.layers.BatchNormalization(name="bn3"),
        tf.keras.layers.MaxPooling2D((2, 2), name="pool3"),

        # Conv Block 4
        tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu', name="conv4"),
        tf.keras.layers.BatchNormalization(name="bn4"),
        tf.keras.layers.MaxPooling2D((2, 2), name="pool4"),

        # Dense Classification Head
        tf.keras.layers.Flatten(name="flatten"),
        tf.keras.layers.Dense(512, activation='relu', name="dense1"),
        tf.keras.layers.Dropout(0.5, name="dropout"),
        tf.keras.layers.Dense(num_classes, activation='softmax', name="predictions")
    ], name="AgriLeaf_CNN")
    return model

model = build_crop_cnn(input_shape=(224, 224, 3), num_classes=NUM_CLASSES)
model.summary()


## 9. Model Compilation

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


## 10. Training Callbacks (EarlyStopping, Checkpoint, ReduceLROnPlateau)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_crop_disease_model.keras",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]


## 11. Model Training Execution

In [ ]:
# history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS,
#     callbacks=callbacks
# )
print("Execute model.fit(...) when running in Google Colab with GPU.")


## 12. Training Accuracy & Loss Graphs

In [ ]:
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='#15803D', lw=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='#0284C7', lw=2)
    axes[0].set_title('Training vs Validation Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Loss
    axes[1].plot(history.history['loss'], label='Train Loss', color='#DC2626', lw=2)
    axes[1].plot(history.history['val_loss'], label='Validation Loss', color='#D97706', lw=2)
    axes[1].set_title('Training vs Validation Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Cross-Entropy Loss')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

# plot_training_history(history)


## 13. Confusion Matrix Evaluation

In [ ]:
def evaluate_confusion_matrix(model, test_dataset, class_names):
    y_true = []
    y_pred = []
    for images, labels in test_dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(18, 14))
    sns.heatmap(cm, annot=False, cmap='Greens', fmt='d',
                xticklabels=[c.split('___')[-1] for c in class_names],
                yticklabels=[c.split('___')[-1] for c in class_names])
    plt.title('Test Partition Confusion Matrix (38 Classes)', fontsize=14)
    plt.xlabel('Predicted Class')
    plt.ylabel('Ground Truth Class')
    plt.xticks(rotation=90)
    plt.show()
    return y_true, y_pred


## 14. Classification Report (Precision, Recall, F1)

In [ ]:
# y_true, y_pred = evaluate_confusion_matrix(model, test_ds, class_names)
# print(classification_report(y_true, y_pred, target_names=class_names))


## 15. Single-Image Test Prediction

In [ ]:
def predict_single_leaf(image_path, model, class_names):
    img = Image.open(image_path).convert('RGB').resize((224, 224))
    img_array = np.expand_dims(np.array(img, dtype=np.float32), axis=0)
    probs = model.predict(img_array, verbose=0)[0]
    top_idx = np.argmax(probs)
    print(f"Predicted: {class_names[top_idx]} ({probs[top_idx]*100:.2f}%)")
    plt.imshow(img)
    plt.title(f"{class_names[top_idx]} ({probs[top_idx]*100:.1f}%)")
    plt.axis('off')
    plt.show()


## 16 & 17. Save Trained Model and Export Backend Assets

In [ ]:
# Save trained model in both .keras and legacy .h5 formats
model.save("crop_disease_model.keras")
model.save("crop_disease_model.h5")

# Export class names
with open("class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)

print("Export complete!")
print("Files ready to be copied into AgriLeaf/backend/model/:")
print("- crop_disease_model.keras")
print("- class_names.json")
